In [0]:
dbutils.widgets.text("api_endpoint", "https://jsonplaceholder.typicode.com/posts")
dbutils.widgets.text("storage_name", "miniproject123")
dbutils.widgets.text("key_scope", "secret-scope")

In [0]:
api_endpoint = dbutils.widgets.get("api_endpoint")
storage_name = dbutils.widgets.get("storage_name")
key_scope    = dbutils.widgets.get("key_scope")

# 4. Print values to verify
print(f"Target API Endpoint: {api_endpoint}")
print(f"Target Storage Account: {storage_name}")
print(f"Target Key Vault Scope: {key_scope}")

In [0]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def fetch_api_data_with_retry(url: str):
    retry_strategy = Retry(
        total=3,                        # Total number of retries
        backoff_factor=2,               # Exponential wait time: 2s, 4s, 8s...
        status_forcelist=[429, 500, 502, 503, 504], # Retry on these status codes
        allowed_methods=["GET"]
    )

    # 2. Mount HTTP adapter with session
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    # 3. Execute GET request with a timeout
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()  # Raise exception for bad HTTP status
        print(f"Successfully fetched data from API. Status Code: {response.status_code}")
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch data from API: {e}")
        raise e
# Fetch payload from the widget endpoint
raw_data = fetch_api_data_with_retry(api_endpoint)

In [0]:
from pyspark.sql import functions as F
# 3. Configure ADLS Gen2 Authentication via Key Vault Secrets
tenant_id     = dbutils.secrets.get(scope=key_scope, key="client-tenant")
client_id     = dbutils.secrets.get(scope=key_scope, key="client-secret")
client_secret = dbutils.secrets.get(scope=key_scope, key="client-value")

spark.conf.set(f"fs.azure.account.auth.type.{storage_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
df_raw = spark.createDataFrame(raw_data)
df_raw = df_raw.withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_system", F.lit("json_placeholder_api"))

bronze_path = f"abfss://bronze@{storage_name}.dfs.core.windows.net/ingested_posts"
df_raw.write.mode("overwrite").format('delta').save(bronze_path)

In [0]:
spark.read.option("mergeSchema", "true").format("delta").load(bronze_path).display()